# HW6: VGG11 on ImageNet Subset (20 classes)



In [1]:
import os, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
TRAIN_DIR, TRAIN_TXT = 'imagenet_train20a', 'imagenet_train20.txt'
VAL_DIR, VAL_TXT = 'imagenet_val20', 'imagenet_val20.txt'

## Data

Labels in the txt files are already class indices 0 to 19. Random crop and horizontal flip are used as data augmentation.

In [2]:
def read_txt(path):
    return [(l.split()[0], int(l.split()[1])) for l in open(path) if l.strip()]

class ImgData(Dataset):
    def __init__(self, root, txt, tf, sub=False):
        # train images sit in class subfolders named by the file prefix
        self.items = [(os.path.join(root, f.split('_')[0] if sub else '', f), y) for f, y in read_txt(txt)]
        self.tf = tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        f, y = self.items[i]
        return self.tf(Image.open(f).convert('RGB')), y

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([transforms.RandomResizedCrop(224), transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
val_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), norm])

train_loader = DataLoader(ImgData(TRAIN_DIR, TRAIN_TXT, train_tf, sub=True), batch_size=32, shuffle=True)
val_loader = DataLoader(ImgData(VAL_DIR, VAL_TXT, val_tf), batch_size=32)
print(len(train_loader.dataset), len(val_loader.dataset))

6000 1000


## 1. Build VGG11

Layers follow the given table: 8 convolution layers (3x3, stride 1, same padding, ReLU), 5 max pooling layers (2x2, stride 2), flatten to 25088, then FC 4096, FC 4096, FC 20. Softmax is applied inside `nn.CrossEntropyLoss`, and the predicted class in [0, 19] is the argmax of the output.

I also added Dropout (p=0.5) after each of the first two FC layers. This is not in the table, but it is part of the standard VGG design and matches the pretrained model, so the loaded weights fit the layout. (The table lists the 4th max pool output as 28x28x256. It should be 14x14x512, and that is what this code produces.)

In [3]:
class VGG11(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()
        layers, c_in = [], 3
        for v in [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']:
            if v == 'M':
                layers.append(nn.MaxPool2d(2, 2))
            else:
                layers += [nn.Conv2d(c_in, v, 3, 1, padding=1), nn.ReLU(inplace=True)]
                c_in = v
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True), nn.Dropout(),
            nn.Linear(4096, 4096), nn.ReLU(inplace=True), nn.Dropout(),
            nn.Linear(4096, num_classes))

    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

model = VGG11()
print(model)

VGG11(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)

**Weight initialization:** weights are initialized from ImageNet pretrained VGG11. The last fully connected layer (1000 outputs) is not loaded, so our new 20 output layer keeps its random initialization. The printed result below confirms that only `classifier.6.weight` and `classifier.6.bias` are missing, so every other layer loaded the pretrained weights.

In [4]:
pre = models.vgg11(weights='IMAGENET1K_V1').state_dict()
pre = {k: v for k, v in pre.items() if not k.startswith('classifier.6')}
print(model.load_state_dict(pre, strict=False))
model = model.to(device)

_IncompatibleKeys(missing_keys=['classifier.6.weight', 'classifier.6.bias'], unexpected_keys=[])


## 2. Training with SGD

Hyperparameters: batch size 32, learning rate 0.001, momentum 0.9, 10 epochs. Train and validation accuracy are printed every epoch.

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

def accuracy(loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
    return correct / len(loader.dataset)

for epoch in range(10):
    model.train()
    correct = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == y).sum().item()
    print(f'Epoch {epoch+1}: train acc = {correct/len(train_loader.dataset):.4f}, val acc = {accuracy(val_loader):.4f}')

Epoch 1: train acc = 0.8292, val acc = 0.9650
Epoch 2: train acc = 0.9122, val acc = 0.9690
Epoch 3: train acc = 0.9167, val acc = 0.9620
Epoch 4: train acc = 0.9220, val acc = 0.9640
Epoch 5: train acc = 0.9300, val acc = 0.9760
Epoch 6: train acc = 0.9395, val acc = 0.9760
Epoch 7: train acc = 0.9363, val acc = 0.9720
Epoch 8: train acc = 0.9392, val acc = 0.9730
Epoch 9: train acc = 0.9447, val acc = 0.9700
Epoch 10: train acc = 0.9400, val acc = 0.9690


## Results

After 10 epochs: **train accuracy 94.0%, validation accuracy 96.9%** (both above the 50% target). Settings: SGD with lr 0.001 and momentum 0.9, batch size 32, 10 epochs, ImageNet pretrained initialization, RandomResizedCrop and horizontal flip augmentation.

Validation accuracy is higher than training accuracy because they are measured under different conditions. Training accuracy is computed during training, with random crops, flips, and dropout all active. Validation accuracy is computed in eval mode, with dropout off and a clean center crop. The model is not overfitting.